## analyzer class

In [ ]:
"""
Improved Climate Report Analyzer - Clean Version
- Better PDF extraction with table detection
- Enhanced noise filtering
- Clean chunk boundaries (no mid-sentence breaks)
- Fast NLLB translation
"""

import json
import re
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union
from datetime import datetime
import time
from collections import Counter
import pdfplumber
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm
import langid
import random
import spacy


class ClimateReportAnalyzer:
    # Chunking settings
    MIN_CHARS = 400
    MAX_CHARS = 2000
    MAX_TOKENS = MAX_CHARS // 4
    BATCH_SIZE = 48

    # Translation settings
    TRANSLATE_BATCH_SIZE = 48
    TRANSLATE_INPUT_MAX_TOKENS = 400
    TRANSLATE_OUTPUT_MAX_TOKENS = 400

    def __init__(self, cache_dir="cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        self.device = 0 if torch.cuda.is_available() else -1
        self.models = {}
        self.translator = None

        # Load spacy for sentence splitting
        self.nlp = None
        try:
            self.nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
        except OSError:
            print("⚠ Spacy model not found. Download with: python -m spacy download en_core_web_sm")

        # NLLB language mapping
        self.nllb_lang_map = {
            'de': 'deu_Latn', 'fr': 'fra_Latn', 'es': 'spa_Latn',
            'it': 'ita_Latn', 'pt': 'por_Latn', 'nl': 'nld_Latn',
            'pl': 'pol_Latn', 'sv': 'swe_Latn', 'fi': 'fin_Latn',
            'no': 'nob_Latn', 'nn': 'nno_Latn', 'da': 'dan_Latn',
            'ru': 'rus_Cyrl', 'uk': 'ukr_Cyrl', 'zh': 'zho_Hans',
            'ja': 'jpn_Jpan', 'ko': 'kor_Hang', 'ar': 'arb_Arab',
            'tr': 'tur_Latn', 'cs': 'ces_Latn', 'ro': 'ron_Latn',
            'hu': 'hun_Latn', 'en': 'eng_Latn',
        }

    # ========================================================================
    # STEP 1: EXTRACT PDF TEXT
    # ========================================================================

    def extract_pdf(self, pdf_path: str) -> Dict:
        """Extract text from PDF and chunk it intelligently"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_extracted.json"

        if cache_file.exists():
            print(f"✓ Loading cached extraction for {pdf_stem}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"\n{'='*80}")
        print(f"STEP 1: EXTRACTING PDF")
        print(f"{'='*80}\n")

        text = self._extract_text_from_pdf(pdf_path)
        print(f"✓ Extracted {len(text):,} characters")

        lang = self._detect_language(text)
        chunks = self._chunk_text(text)
        print(f"✓ Created {len(chunks)} chunks")

        result = {
            "pdf_path": str(pdf_path),
            "language": lang,
            "num_chunks": len(chunks),
            "chunks": chunks,
            "extracted_at": str(datetime.now())
        }

        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"✓ Cached to {cache_file.name}\n")
        return result

    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract and preprocess text from PDF"""
        raw_text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages, desc="Reading pages"):
                page_text = page.extract_text()
                if page_text:
                    raw_text += page_text + "\n"

        text = self._preprocess_text(raw_text)
        return text

    def _preprocess_text(self, text: str) -> str:
        """
        Enhanced text cleaning:
        1. Unicode normalization
        2. Remove hyphenation
        3. Filter noise (headers, footers, navigation)
        4. Reconstruct paragraphs
        """
        # Stage 1: Unicode normalization
        text = unicodedata.normalize('NFC', text)

        # Stage 2: Fix hyphenated line breaks
        text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)

        # Stage 3: Remove zero-width characters
        for char in ['\u00ad', '\u200b', '\u200c', '\u200d', '\ufeff']:
            text = text.replace(char, '')

        # Stage 4: Process line by line
        lines = text.split('\n')

        # Identify repeated lines (likely headers/footers)
        line_counts = Counter(line.strip() for line in lines if line.strip())
        repeated_lines = {line for line, count in line_counts.items()
                         if count >= 3 and len(line) < 100}

        cleaned_lines = []
        for line in lines:
            line = line.strip()

            if not line or line in repeated_lines or self._is_noise(line):
                continue

            # Normalize whitespace
            line = re.sub(r'\s+', ' ', line)
            cleaned_lines.append(line)

        # Stage 5: Reconstruct paragraphs
        paragraphs = self._reconstruct_paragraphs(cleaned_lines)

        # Stage 6: Join with double newlines
        text = '\n\n'.join(paragraphs)
        text = re.sub(r'\n{3,}', '\n\n', text)

        return text.strip()

    def _is_noise(self, line: str) -> bool:
        """Detect headers, footers, page numbers, navigation"""
        line = line.strip()

        if len(line) < 3:
            return True

        # Pure numbers (page numbers)
        if re.match(r'^\d+$', line):
            return True

        # Roman numerals
        if re.match(r'^[ivxlcdm]+$', line.lower()) and len(line) <= 10:
            return True

        # Navigation/TOC keywords
        noise_keywords = [
            'inhalt', 'inhaltsverzeichnis',
            'geschäftsbericht', 'jahresbericht',
            'seite', 'siehe auch', 'vgl.',
        ]

        line_lower = line.lower()
        if any(keyword in line_lower for keyword in noise_keywords):
            if len(line.split()) < 5:  # Short lines only
                return True

        # Common patterns
        patterns = [
            r'^page\s+\d+', r'^\d+\s+of\s+\d+',
            r'^\d{1,2}[/-]\d{1,2}[/-]\d{2,4}$',
            r'^©.*\d{4}', r'^http[s]?://', r'^www\.',
        ]

        for pattern in patterns:
            if re.search(pattern, line, re.IGNORECASE):
                return True

        # TOC leaders (lots of dots/dashes)
        if line.count('.') > len(line) * 0.4 or line.count('-') > len(line) * 0.4:
            return True

        # Short uppercase lines (headers without content)
        if len(line) < 40 and sum(1 for c in line if c.isupper()) > len(line) * 0.7:
            words = line.split()
            if len(words) <= 3:
                return True

        return False

    def _reconstruct_paragraphs(self, lines: List[str]) -> List[str]:
        """Reconstruct paragraphs, handling tables specially"""
        paragraphs = []
        current_para = ""
        in_table = False
        table_lines = []

        for i, line in enumerate(lines):
            is_table_row = self._is_table_row(line)

            if is_table_row:
                if not in_table and current_para:
                    paragraphs.append(current_para.strip())
                    current_para = ""
                in_table = True
                table_lines.append(line)
            else:
                if in_table:
                    if table_lines:
                        paragraphs.append(" ".join(table_lines))
                        table_lines = []
                    in_table = False

                should_break = self._should_break_paragraph(line, current_para, i, lines)

                if should_break and current_para:
                    paragraphs.append(current_para.strip())
                    current_para = line
                else:
                    if current_para:
                        current_para += " " + line
                    else:
                        current_para = line

        if in_table and table_lines:
            paragraphs.append(" ".join(table_lines))
        if current_para:
            paragraphs.append(current_para.strip())

        return [p for p in paragraphs if len(p) > 100]

    def _is_table_row(self, line: str) -> bool:
        """Detect table rows (multiple numbers)"""
        numbers = re.findall(r'\d+[,.]?\d*', line)
        words = line.split()

        if len(numbers) >= 3 and len(words) > 0:
            if len(numbers) / len(words) > 0.4:
                return True

        return False

    def _should_break_paragraph(self, line: str, current_para: str,
                                idx: int, all_lines: List[str]) -> bool:
        """Determine if we should start a new paragraph"""
        if not current_para:
            return False

        # Break if current paragraph is long
        if len(current_para) > 800:
            return True

        # Check if previous line ended with sentence-ending punctuation
        if current_para and current_para[-1] in '.!?':
            words = current_para.split()
            if words and len(words[-1]) > 3:  # Not abbreviation
                return True

        # Break on bullets or numbering
        if re.match(r'^[\-•●○▪▫◦]\s', line) or re.match(r'^\d+[\.\)]\s', line):
            return True

        # Break on short uppercase lines (section headers)
        if len(line) < 60 and line.isupper():
            return True

        return False

    def _detect_language(self, text: str) -> str:
        """Detect language using langid"""
        try:
            sample = text[:5000]
            lang, confidence = langid.classify(sample)
            print(f"✓ Language detected: {lang} (confidence: {confidence:.2f})")
            return lang
        except Exception as e:
            print(f"⚠ Language detection failed: {e}. Defaulting to 'en'")
            return 'en'

    def _chunk_text(self, text: str) -> List[str]:
        """
        Smart chunking with clean boundaries:
        - Split by paragraphs first
        - Use sentence boundaries for large paragraphs
        - Never end chunks with commas or conjunctions
        """
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        current_chunk = []
        current_size = 0

        for para in paragraphs:
            para_size = len(para)

            if current_size + para_size > self.MAX_CHARS and current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunk_text = self._clean_chunk_ending(chunk_text)
                chunks.append(chunk_text)
                current_chunk = []
                current_size = 0

            if para_size > self.MAX_CHARS:
                sentences = self._split_sentences(para)
                for sent in sentences:
                    sent_size = len(sent)

                    if current_size + sent_size > self.MAX_CHARS and current_chunk:
                        chunk_text = ' '.join(current_chunk)
                        chunk_text = self._clean_chunk_ending(chunk_text)
                        chunks.append(chunk_text)
                        current_chunk = [sent]
                        current_size = sent_size
                    else:
                        current_chunk.append(sent)
                        current_size += sent_size
            else:
                current_chunk.append(para)
                current_size += para_size

        if current_chunk:
            chunk_text = ' '.join(current_chunk)
            chunk_text = self._clean_chunk_ending(chunk_text)
            chunks.append(chunk_text)

        valid_chunks = [c for c in chunks if len(c) >= self.MIN_CHARS]

        print(f"  Created {len(chunks)} chunks, kept {len(valid_chunks)} "
              f"(filtered {len(chunks) - len(valid_chunks)} too short)")

        return valid_chunks

    def _clean_chunk_ending(self, text: str) -> str:
        """Ensure chunk doesn't end badly"""
        text = text.strip()

        bad_endings = ['und', 'and', 'or', 'oder', 'but', 'aber', 'sowie']

        words = text.split()
        if not words:
            return text

        last_word = words[-1].rstrip('.,;:')

        # If ends with comma or conjunction, cut back to last sentence
        if last_word.lower() in bad_endings or text[-1] == ',':
            sentences = self._split_sentences(text)
            if len(sentences) > 1:
                return ' '.join(sentences[:-1])

        return text

    def _split_sentences(self, text: str) -> List[str]:
        """Split text into sentences"""
        if self.nlp is not None:
            doc = self.nlp(text)
            return [sent.text.strip() for sent in doc.sents]
        else:
            # Improved fallback regex
            sentences = re.split(
                r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<![A-Z]\.)(?<=\.|\?|\!)\s+',
                text
            )
            return [s.strip() for s in sentences if s.strip()]

    # ========================================================================
    # STEP 2: TRANSLATE
    # ========================================================================

    def translate_pdf(self, pdf_path: str) -> Dict:
        """Translate extracted chunks to English"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_translated.json"

        if cache_file.exists():
            print(f"✓ Loading cached translation for {pdf_stem}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"\n{'='*80}")
        print(f"STEP 2: TRANSLATING")
        print(f"{'='*80}\n")

        extracted = self.extract_pdf(pdf_path)
        lang = extracted['language']
        chunks = extracted['chunks']

        if lang == 'en':
            print(f"✓ Already in English")
            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": False,
                "chunk_pairs": [{"original": c, "translated": c} for c in chunks],
                "translated_at": str(datetime.now())
            }
        elif lang not in self.nllb_lang_map:
            print(f"⚠ Translation not supported for: {lang}")
            print(f"  Supported: {list(self.nllb_lang_map.keys())}")
            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": False,
                "unsupported": True,
                "chunk_pairs": [{"original": c, "translated": c} for c in chunks],
                "translated_at": str(datetime.now())
            }
        else:
            start = time.time()
            translated_chunks = self._translate_chunks_nllb(chunks, lang)
            elapsed = time.time() - start

            print(f"✓ Translated {len(translated_chunks)} chunks in {elapsed:.1f}s "
                  f"({elapsed/len(chunks):.2f}s per chunk)")

            detected = langid.classify(translated_chunks[0][:500])[0]
            print(f"✓ First chunk detected as: {detected}\n")

            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": True,
                "chunk_pairs": [
                    {"original": orig, "translated": trans}
                    for orig, trans in zip(chunks, translated_chunks)
                ],
                "translated_at": str(datetime.now())
            }

        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"✓ Cached to {cache_file.name}\n")
        return result

    def _load_translator(self):
        """Load NLLB model once for all languages"""
        if self.translator is None:
            print(f"Loading NLLB-200 translation model...")

            model_name = "facebook/nllb-200-distilled-600M"
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

            if self.device >= 0:
                model = model.to('cuda')
                try:
                    model = model.half()  # FP16 for 2x speed
                    print(f"✓ Model loaded on GPU with FP16")
                except:
                    print(f"✓ Model loaded on GPU (FP32)")
            else:
                print(f"✓ Model loaded on CPU (slow)")

            model.eval()

            self.translator = {
                'model': model,
                'tokenizer': tokenizer
            }
        return self.translator

    def _translate_chunks_nllb(self, chunks: List[str], src_lang: str) -> List[str]:
        """Fast batch translation using NLLB"""
        translator = self._load_translator()
        model = translator['model']
        tokenizer = translator['tokenizer']

        src_code = self.nllb_lang_map[src_lang]
        tgt_code = self.nllb_lang_map['en']

        print(f"Translating from {src_lang} ({src_code}) → en ({tgt_code})")
        print(f"Batch size: {self.TRANSLATE_BATCH_SIZE}, Device: {'GPU' if self.device >= 0 else 'CPU'}")

        translated = []

        with torch.no_grad():
            for i in tqdm(range(0, len(chunks), self.TRANSLATE_BATCH_SIZE), desc="Translating"):
                batch = chunks[i:i+self.TRANSLATE_BATCH_SIZE]

                try:
                    tokenizer.src_lang = src_code

                    inputs = tokenizer(
                        batch,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=self.TRANSLATE_INPUT_MAX_TOKENS
                    )

                    if self.device >= 0:
                        inputs = {k: v.to('cuda') for k, v in inputs.items()}

                    tgt_token_id = tokenizer.convert_tokens_to_ids(tgt_code)

                    translated_tokens = model.generate(
                        **inputs,
                        forced_bos_token_id=tgt_token_id,
                        max_length=self.TRANSLATE_OUTPUT_MAX_TOKENS,
                        num_beams=1,
                        do_sample=False,
                        use_cache=True
                    )

                    batch_translations = tokenizer.batch_decode(
                        translated_tokens,
                        skip_special_tokens=True
                    )

                    translated.extend(batch_translations)

                except Exception as e:
                    print(f"\n⚠ Translation error: {type(e).__name__}: {e}")
                    translated.extend(batch)

        return translated

    # ========================================================================
    # INSPECTION
    # ========================================================================

    def inspect_extraction(self, pdf_path: str, n_samples=3):
        """Show extracted chunks with statistics"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_extracted.json"

        if not cache_file.exists():
            print(f"No extracted data. Run extract_pdf() first.")
            return

        with open(cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        chunks = data['chunks']
        chunk_lengths = [len(c) for c in chunks]

        print(f"\n{'='*80}")
        print(f"EXTRACTION: {Path(pdf_path).name}")
        print(f"Language: {data['language']}")
        print(f"Total chunks: {data['num_chunks']}")
        print(f"Avg size: {sum(chunk_lengths)/len(chunk_lengths):.0f} chars")
        print(f"Min/Max: {min(chunk_lengths)} / {max(chunk_lengths)} chars")
        print(f"{'='*80}\n")

        samples = random.sample(chunks, min(n_samples, len(chunks)))

        for i, chunk in enumerate(samples, 1):
            print(f"{'─'*80}")
            print(f"Chunk {i} ({len(chunk)} chars)")
            print(f"{'─'*80}")
            print(f"{chunk[:500]}{'...' if len(chunk) > 500 else ''}\n")

    def inspect_translation(self, pdf_path: str, n_samples=2):
        """Show original vs translated"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_translated.json"

        if not cache_file.exists():
            print(f"No translation data. Run translate_pdf() first.")
            return

        with open(cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f"\n{'='*80}")
        print(f"TRANSLATION: {Path(pdf_path).name}")
        print(f"Language: {data['language']} → en")
        print(f"Translated: {data.get('translated', False)}")
        print(f"Total chunks: {len(data['chunk_pairs'])}")
        print(f"{'='*80}\n")

        samples = random.sample(data['chunk_pairs'], min(n_samples, len(data['chunk_pairs'])))

        for i, pair in enumerate(samples, 1):
            orig = pair['original']
            trans = pair['translated']

            print(f"{'─'*80}")
            print(f"Sample {i}")
            print(f"{'─'*80}")
            print(f"\nORIGINAL ({data['language']}, {len(orig)} chars):")
            print(f"{orig[:400]}{'...' if len(orig) > 400 else ''}")

            if orig != trans:
                print(f"\nTRANSLATED (en, {len(trans)} chars):")
                print(f"{trans[:400]}{'...' if len(trans) > 400 else ''}")

            print()

    # ========================================================================
    # ANALYSIS
    # ========================================================================

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""
        if task_name not in self.models:
            print(f"Loading {task_name} model...")
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.models[task_name] = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=self.device
            )
        return self.models[task_name]

    def get_chunks_for_analysis(self, pdf_path: str, use_translated: bool = True) -> List[str]:
        """Get chunks ready for analysis"""
        pdf_stem = Path(pdf_path).stem
        trans_cache = self.cache_dir / f"{pdf_stem}_translated.json"
        extract_cache = self.cache_dir / f"{pdf_stem}_extracted.json"

        if use_translated and trans_cache.exists():
            with open(trans_cache, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return [pair['translated'] for pair in data['chunk_pairs']]

        if extract_cache.exists():
            with open(extract_cache, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data['chunks']

        extracted = self.extract_pdf(pdf_path)
        return extracted['chunks']


    def filter_climate_chunks(self, data: Union[Dict, str], batch_size=None) -> Dict:
        """Filter for climate-related chunks with caching

        Args:
            data: Either translated JSON dict, path to translated JSON, or PDF path
            batch_size: Batch size for processing

        Returns:
            Dict with filtered chunks and metadata
        """
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        # Handle input - get PDF path and load data
        if isinstance(data, str):
            pdf_path = Path(data)
            # Check for filtered cache first
            cache_file = self.cache_dir / f"{pdf_path.stem}_filtered.json"

            if cache_file.exists():
                print(f"✓ Loading filtered chunks from cache: {cache_file.name}")
                with open(cache_file, 'r', encoding='utf-8') as f:
                    return json.load(f)

            # Try to load translated data first, fall back to extracted
            translated_cache = self.cache_dir / f"{pdf_path.stem}_translated.json"
            extracted_cache = self.cache_dir / f"{pdf_path.stem}_extracted.json"

            if translated_cache.exists():
                with open(translated_cache, 'r', encoding='utf-8') as f:
                    data = json.load(f)
            elif extracted_cache.exists():
                with open(extracted_cache, 'r', encoding='utf-8') as f:
                    data = json.load(f)
            else:
                raise FileNotFoundError(f"No extracted or translated data found for {pdf_path.name}")
        elif isinstance(data, dict):
            # If dict passed, we can't cache (no path known)
            pdf_path = None
            cache_file = None
        else:
            raise TypeError(f"Expected dict or str (path), got {type(data)}")

        # Extract text chunks from data structure
        if "chunk_pairs" in data:
            # Translated format - use translated text
            chunks = [pair["translated"] for pair in data["chunk_pairs"]]
            source = "translated"
        elif "chunks" in data:
            # Extracted format - use original text
            chunks = data["chunks"]
            source = "original"
        else:
            raise ValueError("Data must have 'chunk_pairs' or 'chunks' key")

        # Load detector model
        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} {source} chunks for climate content...")
        climate_chunks = []

        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting"):
            batch = chunks[i:i+batch_size]

            try:
                results = detector(batch, truncation=True, max_length=512,
                                batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score'],
                            'detector_label': result['label']
                        })
            except Exception as e:
                print(f"⚠ Error in batch {i}: {e}")
                continue

        kept_pct = len(climate_chunks)/len(chunks)*100 if chunks else 0
        print(f"✓ Kept {len(climate_chunks)} climate chunks ({kept_pct:.1f}%)")

        # Build result
        result = {
            'pdf_path': str(pdf_path) if pdf_path else data.get('pdf_path', 'unknown'),
            'language': data.get('language', 'unknown'),
            'source': source,  # Track whether from translation or original
            'filtered': True,
            'filtered_at': datetime.now().isoformat(),
            'total_chunks': len(chunks),
            'climate_chunks': len(climate_chunks),
            'kept_percentage': kept_pct,
            'chunks': climate_chunks
        }

        # Cache if we have a path
        if cache_file:
            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            print(f"✓ Cached filtered chunks: {cache_file.name}")

        return result




    # ========================================================================
    # BATCH PROCESSING
    # ========================================================================
    def process_pdfs(self, path: str, extract: bool = True,
                translate: bool = True, filter_climate: bool = True,
                skip_errors: bool = True):
        """
        Process PDF(s) - handles both single files and directories

        Args:
            path: Path to PDF file or directory
            extract: Whether to extract text
            translate: Whether to translate (requires extract=True)
            filter_climate: Whether to filter for climate-related chunks
            skip_errors: Continue on errors instead of stopping (only for directories)

        Returns:
            Dict with statistics and any errors encountered
        """
        target = Path(path)

        if not target.exists():
            print(f"❌ Path not found: {path}")
            return None

        # Determine if single file or directory
        if target.is_file():
            if target.suffix.lower() != '.pdf':
                print(f"❌ Not a PDF file: {path}")
                return None
            pdf_files = [target]
            is_single = True
            root = target.parent
        else:
            # Find all PDFs recursively
            pdf_files = sorted(target.rglob("*.pdf"))
            if not pdf_files:
                print(f"❌ No PDFs found in {path}")
                return None
            is_single = False
            root = target

        # Print header
        print(f"\n{'='*80}")
        if is_single:
            print(f"PROCESSING SINGLE PDF")
        else:
            print(f"BATCH PROCESSING")
        print(f"{'='*80}")

        if is_single:
            print(f"File: {pdf_files[0].name}")
        else:
            print(f"Found {len(pdf_files)} PDFs in {path}")

        print(f"Extract: {extract}, Translate: {translate}, Filter: {filter_climate}")
        print(f"{'='*80}\n")

        stats = {
            'total': len(pdf_files),
            'extracted': 0,
            'translated': 0,
            'filtered': 0,
            'skipped_translation': 0,
            'errors': []
        }

        for i, pdf_path in enumerate(pdf_files, 1):
            relative_path = pdf_path.relative_to(root)

            if is_single:
                print(f"Processing: {relative_path}")
            else:
                print(f"\n[{i}/{len(pdf_files)}] Processing: {relative_path}")

            try:
                if extract:
                    extracted = self.extract_pdf(str(pdf_path))
                    stats['extracted'] += 1

                    if translate:
                        translated = self.translate_pdf(str(pdf_path))

                        if translated.get('translated', False):
                            stats['translated'] += 1
                            print(f"  ✓ Translated: {translated['language']} → en")
                        else:
                            stats['skipped_translation'] += 1
                            print(f"  ⊳ Already English, skipped translation")

                    # Filter climate chunks regardless of translation
                    if filter_climate:
                        filtered = self.filter_climate_chunks(str(pdf_path))
                        stats['filtered'] += 1
                        print(f"  ✓ Filtered: {filtered['climate_chunks']}/{filtered['total_chunks']} chunks "
                            f"({filtered['kept_percentage']:.1f}%)")

                print(f"  ✓ Done")

            except Exception as e:
                error_info = {
                    'file': str(relative_path),
                    'error': str(e),
                    'type': type(e).__name__
                }
                stats['errors'].append(error_info)

                if is_single or not skip_errors:
                    print(f"  ❌ Error: {type(e).__name__}: {e}")
                    raise
                else:
                    print(f"  ⚠ Error (skipping): {type(e).__name__}: {e}")

        # Print summary
        print(f"\n{'='*80}")
        if is_single:
            print(f"PROCESSING COMPLETE")
        else:
            print(f"BATCH PROCESSING COMPLETE")
        print(f"{'='*80}")
        print(f"Total PDFs:          {stats['total']}")
        print(f"Extracted:           {stats['extracted']}")
        print(f"Translated:          {stats['translated']}")
        print(f"Skipped (English):   {stats['skipped_translation']}")
        print(f"Filtered:            {stats['filtered']}")
        print(f"Errors:              {len(stats['errors'])}")

        if stats['errors']:
            print(f"\n⚠ Errors encountered:")
            for err in stats['errors']:
                print(f"  • {err['file']}: {err['type']}")

        print(f"{'='*80}\n")

        return stats

    def list_pdfs(self, root_dir: str, show_cached: bool = True):
        """
        List all PDFs and their cache status

        Args:
            root_dir: Directory to search
            show_cached: Show which PDFs already have cache files
        """

        root = Path(root_dir)
        pdf_files = sorted(root.rglob("*.pdf"))

        if not pdf_files:
            print(f"No PDFs found in {root_dir}")
            return

        print(f"\n{'='*80}")
        print(f"PDFs in {root_dir}")
        print(f"{'='*80}\n")

        cached_extract = 0
        cached_translate = 0

        for pdf_path in pdf_files:
            relative = pdf_path.relative_to(root)
            pdf_stem = pdf_path.stem

            extract_cache = self.cache_dir / f"{pdf_stem}_extracted.json"
            translate_cache = self.cache_dir / f"{pdf_stem}_translated.json"

            status = []
            if extract_cache.exists():
                status.append("extracted")
                cached_extract += 1
            if translate_cache.exists():
                status.append("translated")
                cached_translate += 1

            status_str = ", ".join(status) if status else "not cached"

            if show_cached or not status:
                print(f"  {relative}")
                if status:
                    print(f"    └─ {status_str}")

        print(f"\n{'─'*80}")
        print(f"Total: {len(pdf_files)} PDFs")
        print(f"Cached extractions: {cached_extract}")
        print(f"Cached translations: {cached_translate}")
        print(f"{'='*80}\n")

## step by step

In [ ]:
analyzer = ClimateReportAnalyzer()

# pdf_path = "data/reports/Dillinger/012_2019_annual_report.pdf"
pdf_path = "data/reports/LibertySteel/011_2022_sustainability_report.pdf"

analyzer.extract_pdf(pdf_path)
analyzer.translate_pdf(pdf_path)
analyzer.filter_climate_chunks(pdf_path )

# analyzer.inspect_extraction(pdf_path, n_samples=2)
# analyzer.inspect_translation(pdf_path, n_samples=2)
# analyzer.list_pdfs(Path(pdf_path).parent)
# print(type(translated))
print(filtered)



## whole pipe

In [80]:
# run pipeline auto (extract+translate)
analyzer = ClimateReportAnalyzer()

pdf_path="data/reports/Celsa"

analyzer.list_pdfs(pdf_path) # list pdfs + cached


# extract + translate pdf # 205min for 144pdfs
stats = analyzer.process_pdfs(
    path=pdf_path,
    extract=True,
    translate=True,
    filter_climate=True,
    skip_errors=True  # Continue even if some PDFs fail
)

# Check the results
print(f"Processed {stats['extracted']} PDFs")
print(f"Translated {stats['translated']} PDFs")
if stats['errors']:
    print(f"Failed on {len(stats['errors'])} PDFs")


PDFs in data/reports/Celsa

  007_2020_sustainability_report.pdf
    └─ extracted, translated
  007_2021_sustainability_report.pdf
    └─ extracted, translated
  007_2022_sustainability_report.pdf
    └─ extracted, translated
  007_2023_sustainability_report.pdf
    └─ extracted, translated
  007_2024_sustainability_report.pdf
    └─ extracted, translated

────────────────────────────────────────────────────────────────────────────────
Total: 5 PDFs
Cached extractions: 5
Cached translations: 5


BATCH PROCESSING
Found 5 PDFs in data/reports/Celsa
Extract: True, Translate: True, Filter: True


[1/5] Processing: 007_2020_sustainability_report.pdf
✓ Loading cached extraction for 007_2020_sustainability_report
✓ Loading cached translation for 007_2020_sustainability_report
  ✓ Translated: es → en
✓ Loading filtered chunks from cache: 007_2020_sustainability_report_filtered.json
  ✓ Filtered: 101/160 chunks (63.1%)
  ✓ Done

[2/5] Processing: 007_2021_sustainability_report.pdf
✓ Loading ca

Device set to use cpu



Filtering 104 translated chunks for climate content...


Detecting:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Kept 78 climate chunks (75.0%)
✓ Cached filtered chunks: 007_2021_sustainability_report_filtered.json
  ✓ Filtered: 78/104 chunks (75.0%)
  ✓ Done

[3/5] Processing: 007_2022_sustainability_report.pdf
✓ Loading cached extraction for 007_2022_sustainability_report
✓ Loading cached translation for 007_2022_sustainability_report
  ✓ Translated: es → en
✓ Loading filtered chunks from cache: 007_2022_sustainability_report_filtered.json
  ✓ Filtered: 125/169 chunks (74.0%)
  ✓ Done

[4/5] Processing: 007_2023_sustainability_report.pdf
✓ Loading cached extraction for 007_2023_sustainability_report
✓ Loading cached translation for 007_2023_sustainability_report
  ✓ Translated: es → en
✓ Loading filtered chunks from cache: 007_2023_sustainability_report_filtered.json
  ✓ Filtered: 121/184 chunks (65.8%)
  ✓ Done

[5/5] Processing: 007_2024_sustainability_report.pdf
✓ Loading cached extraction for 007_2024_sustainability_report
✓ Loading cached translation for 007_2024_sustainability_report
  

Detecting:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Kept 70 climate chunks (75.3%)
✓ Cached filtered chunks: 007_2024_sustainability_report_filtered.json
  ✓ Filtered: 70/93 chunks (75.3%)
  ✓ Done

BATCH PROCESSING COMPLETE
Total PDFs:          5
Extracted:           5
Translated:          3
Skipped (English):   2
Filtered:            5
Errors:              0

Processed 5 PDFs
Translated 3 PDFs
